In [25]:
import sqlite3

# Establish connection
conn = sqlite3.connect('students.db')
cursor = conn.cursor()

# Reset tables for a clean run
cursor.execute('DROP TABLE IF EXISTS students')
cursor.execute('DROP TABLE IF EXISTS courses')

# Create courses table
cursor.execute("""
CREATE TABLE courses (
    course_code TEXT PRIMARY KEY,
    description TEXT NOT NULL
)
""")

# Create students table
cursor.execute("""
CREATE TABLE students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    course TEXT NOT NULL,
    year_level INTEGER NOT NULL,
    FOREIGN KEY (course) REFERENCES courses(course_code)
)
""")

# Seed courses
courses_data = [
    ('BS ECE', 'Electronics and Communications Engineering'),
    ('BS EE',  'Electrical Engineering'),
    ('BS COE', 'Computer Engineering'),
    ('BS CE',  'Civil Engineering')
]
cursor.executemany('INSERT INTO courses VALUES (?, ?)', courses_data)

# Seed students
students_data = [
    ('Anna Reyes',                  'BS ECE', 4),
    ('Carlos Dela Cruz',            'BS EE',  2),
    ('Mika Santos',                 'BS CE',  2),
    ('Ralph Samuel B. Villanueva',  'BS ECE', 1),
    ('David Lee',                   'BS CE',  4),
    ('James Yap',                   'BS CE',  1),
    ('Patricia Garcia',             'BS COE', 3),
    ('John Computer Doe',           'BS COE', 2),
    ('Beatrice Almonte',            'BS EE',  3),
    ('Charlie Reyes',               'BS ECE', 3)
]
cursor.executemany(
    'INSERT INTO students (full_name, course, year_level) VALUES (?, ?, ?)',
    students_data
)
conn.commit()

print('Database ready — students.db')
print(f'{len(courses_data)} courses | {len(students_data)} students inserted')

Database ready — students.db
4 courses | 10 students inserted


In [26]:
cursor.execute("""
    SELECT full_name, course FROM students
    WHERE year_level = (SELECT MAX(year_level) FROM students)
""")
print('Students in the highest year level:')
for row in cursor.fetchall():
    print(row)

Students in the highest year level:
('Anna Reyes', 'BS ECE')
('David Lee', 'BS CE')


In [27]:
cursor.execute("""
    SELECT * FROM students
    WHERE course IN (
        SELECT course_code FROM courses
        WHERE description LIKE '%Engineering%'
    )
""")
print('Students enrolled in any Engineering course:')
for row in cursor.fetchall():
    print(row)

Students enrolled in any Engineering course:
(1, 'Anna Reyes', 'BS ECE', 4)
(2, 'Carlos Dela Cruz', 'BS EE', 2)
(3, 'Mika Santos', 'BS CE', 2)
(4, 'Ralph Samuel B. Villanueva', 'BS ECE', 1)
(5, 'David Lee', 'BS CE', 4)
(6, 'James Yap', 'BS CE', 1)
(7, 'Patricia Garcia', 'BS COE', 3)
(8, 'John Computer Doe', 'BS COE', 2)
(9, 'Beatrice Almonte', 'BS EE', 3)
(10, 'Charlie Reyes', 'BS ECE', 3)


In [28]:
cursor.execute("""
    SELECT course, COUNT(*) AS total
    FROM students
    GROUP BY course
    HAVING total > 1
""")
print('Courses with more than one student:')
print(cursor.fetchall())

Courses with more than one student:
[('BS CE', 3), ('BS COE', 2), ('BS ECE', 3), ('BS EE', 2)]


In [29]:
cursor.execute("""
    SELECT *,
    CASE WHEN year_level >= 4 THEN 'Graduating'
         ELSE 'Regular'
    END AS status
    FROM students
""")
print('Students with graduation status:')
for row in cursor.fetchall():
    print(row)

Students with graduation status:
(1, 'Anna Reyes', 'BS ECE', 4, 'Graduating')
(2, 'Carlos Dela Cruz', 'BS EE', 2, 'Regular')
(3, 'Mika Santos', 'BS CE', 2, 'Regular')
(4, 'Ralph Samuel B. Villanueva', 'BS ECE', 1, 'Regular')
(5, 'David Lee', 'BS CE', 4, 'Graduating')
(6, 'James Yap', 'BS CE', 1, 'Regular')
(7, 'Patricia Garcia', 'BS COE', 3, 'Regular')
(8, 'John Computer Doe', 'BS COE', 2, 'Regular')
(9, 'Beatrice Almonte', 'BS EE', 3, 'Regular')
(10, 'Charlie Reyes', 'BS ECE', 3, 'Regular')


In [30]:
cursor.execute("""
    SELECT s.full_name, c.description
    FROM students s
    LEFT JOIN courses c ON s.course = c.course_code
""")
print('All students with course descriptions (LEFT JOIN):')
for row in cursor.fetchall():
    print(row)

All students with course descriptions (LEFT JOIN):
('Anna Reyes', 'Electronics and Communications Engineering')
('Carlos Dela Cruz', 'Electrical Engineering')
('Mika Santos', 'Civil Engineering')
('Ralph Samuel B. Villanueva', 'Electronics and Communications Engineering')
('David Lee', 'Civil Engineering')
('James Yap', 'Civil Engineering')
('Patricia Garcia', 'Computer Engineering')
('John Computer Doe', 'Computer Engineering')
('Beatrice Almonte', 'Electrical Engineering')
('Charlie Reyes', 'Electronics and Communications Engineering')


In [31]:
cursor.execute("""
    SELECT COUNT(*) FROM students
    WHERE full_name LIKE '% % %'
""")
result = cursor.fetchone()
print(f'Count of students with three-part names: {result[0]}')

Count of students with three-part names: 3


In [32]:
cursor.execute("""
    SELECT course, MAX(year_level)
    FROM students
    GROUP BY course
    ORDER BY MAX(year_level) DESC
""")
print('Highest year level per course (descending):')
for row in cursor.fetchall():
    print(row)

Highest year level per course (descending):
('BS ECE', 4)
('BS CE', 4)
('BS EE', 3)
('BS COE', 3)


In [33]:
cursor.execute("""
    SELECT * FROM students
    WHERE rowid % 2 = 0
""")
print('Students with even row IDs:')
for row in cursor.fetchall():
    print(row)

Students with even row IDs:
(2, 'Carlos Dela Cruz', 'BS EE', 2)
(4, 'Ralph Samuel B. Villanueva', 'BS ECE', 1)
(6, 'James Yap', 'BS CE', 1)
(8, 'John Computer Doe', 'BS COE', 2)
(10, 'Charlie Reyes', 'BS ECE', 3)


In [34]:
cursor.execute("""
    SELECT full_name FROM students
    WHERE full_name GLOB '*e*e*'
""")
print("Names with at least two lowercase 'e's:")
for row in cursor.fetchall():
    print(row)

Names with at least two lowercase 'e's:
('Anna Reyes',)
('Ralph Samuel B. Villanueva',)
('David Lee',)
('John Computer Doe',)
('Beatrice Almonte',)
('Charlie Reyes',)


In [35]:
cursor.execute("""
    SELECT full_name, course FROM students
    WHERE NOT EXISTS (
        SELECT 1 FROM courses
        WHERE course_code = students.course
    )
""")
rows = cursor.fetchall()
if not rows:
    print('No discrepancies found — all students have a registered course.')
else:
    print('Students with unregistered courses:')
    for row in rows:
        print(row)

No discrepancies found — all students have a registered course.


In [36]:
cursor.execute("""
    SELECT course, COUNT(*), AVG(year_level)
    FROM students
    GROUP BY course
""")
print('Course stats — count and average year level:')
for row in cursor.fetchall():
    print(f'Course: {row[0]:8} | Students: {row[1]} | Avg Year: {row[2]:.2f}')

Course stats — count and average year level:
Course: BS CE    | Students: 3 | Avg Year: 2.33
Course: BS COE   | Students: 2 | Avg Year: 2.50
Course: BS ECE   | Students: 3 | Avg Year: 2.67
Course: BS EE    | Students: 2 | Avg Year: 2.50


In [37]:
cursor.execute("""
    SELECT full_name, SUBSTR(full_name, 1, 1) AS initial
    FROM students
""")
print('Student initials:')
for name, initial in cursor.fetchall():
    print(f'Initial: {initial}  |  Name: {name}')

Student initials:
Initial: A  |  Name: Anna Reyes
Initial: C  |  Name: Carlos Dela Cruz
Initial: M  |  Name: Mika Santos
Initial: R  |  Name: Ralph Samuel B. Villanueva
Initial: D  |  Name: David Lee
Initial: J  |  Name: James Yap
Initial: P  |  Name: Patricia Garcia
Initial: J  |  Name: John Computer Doe
Initial: B  |  Name: Beatrice Almonte
Initial: C  |  Name: Charlie Reyes


In [38]:
keyword = input('Enter keyword to search: ')
cursor.execute("""
    SELECT * FROM students
    WHERE full_name LIKE ? LIMIT 1
""", ('%' + keyword + '%',))
print('Search result (LIMIT 1):', cursor.fetchone())

Search result (LIMIT 1): None


In [39]:
cursor.execute("""
    SELECT course, COUNT(*)
    FROM students
    GROUP BY course
    ORDER BY COUNT(*) DESC
    LIMIT 1
""")
print('Most popular course:')
print(cursor.fetchall())

Most popular course:
[('BS ECE', 3)]


In [40]:
cursor.execute("""
    SELECT DISTINCT course FROM students
    WHERE full_name LIKE '%a%'
    AND year_level BETWEEN 2 AND 4
""")
print("Unique courses of students whose name contains 'a' (year 2–4):")
print(cursor.fetchall())

Unique courses of students whose name contains 'a' (year 2–4):
[('BS ECE',), ('BS EE',), ('BS CE',), ('BS COE',)]


In [41]:
cursor.execute("""
    SELECT * FROM students
    WHERE LENGTH(full_name) = (SELECT MAX(LENGTH(full_name)) FROM students)
""")
print('Student(s) with the longest name:')
print(cursor.fetchall())

Student(s) with the longest name:
[(4, 'Ralph Samuel B. Villanueva', 'BS ECE', 1)]


In [42]:
cursor.execute("""
    SELECT COUNT(*) FROM students
    WHERE course = 'BS CE'
    AND year_level = (
        SELECT MIN(year_level) FROM students WHERE course = 'BS CE'
    )
""")
print('Count of BS CE students in the lowest year level:')
print(cursor.fetchall())

Count of BS CE students in the lowest year level:
[(1,)]


In [43]:
cursor.execute("""
    SELECT s.full_name, s.year_level, c.description
    FROM students s
    JOIN courses c ON s.course = c.course_code
    WHERE c.description LIKE '%Computer%'
""")
print("Students in courses with 'Computer' in the description:")
for row in cursor.fetchall():
    print(row)

Students in courses with 'Computer' in the description:
('Patricia Garcia', 3, 'Computer Engineering')
('John Computer Doe', 2, 'Computer Engineering')


In [44]:
try:
    cursor.execute('SELECT * FROM unknown_table')
except Exception as e:
    print('Error:', e)

Error: no such table: unknown_table


In [45]:
cursor.execute("""
    SELECT course, COUNT(*) FROM students
    GROUP BY course
    HAVING COUNT(*) = (
        SELECT MAX(cnt)
        FROM (SELECT COUNT(*) AS cnt FROM students GROUP BY course)
    )
""")
print('Course(s) with the highest number of students (ties included):')
print(cursor.fetchall())

Course(s) with the highest number of students (ties included):
[('BS CE', 3), ('BS ECE', 3)]


In [46]:
conn.commit()
conn.close()
print('Database connection closed.')

Database connection closed.
